In [ ]:
import numpy as np

def getLv(ell, K, R):
    """
    Generate a binary matrix L_v of size (K, ell) where approximately 10% of the elements are False.

    Parameters:
    ell (int): Number of columns.
    K (int): Number of rows.
    R (float): Ratio of False values (default: 0.1).

    Returns:
    np.ndarray: Boolean matrix of shape (K, ell).
    """
    L_v = np.ones((K, ell), dtype=bool)  # Initialize with all True

    # Compute the exact number of False values needed
    num_false = int(np.ceil(R * K * ell))

    # Randomly select indices to set as False
    false_indices = np.random.choice(K * ell, size=num_false, replace=False)

    # Convert 1D indices to 2D indices
    row_indices, col_indices = np.unravel_index(false_indices, (K, ell))

    # Set selected positions to False
    L_v[row_indices, col_indices] = False

    return L_v

ell = 500
K = 3
R = 0.9
Lambda = 0.001
L_v=getLv(ell, K, R)

In [ ]:
def is_pos_def(matrix):
    return np.all(np.linalg.eigvals(matrix) > 0)

def get_inc_kerns(kernmats, L_v):
    ell, _, K = kernmats.shape
    Q_inc = np.zeros((ell, ell, K))

    for k in range(K):
        l_tmp1 = L_v[k, :]
        Q_inc[np.ix_(l_tmp1, l_tmp1, [k])] = kernmats[np.ix_(l_tmp1, l_tmp1, [k])]
        if not is_pos_def(Q_inc[:, :, k]):
            Q_inc[:, :, k] += 0.001 * np.eye(ell)

    return Q_inc


In [ ]:
np.random.seed(42)  # For reproducibility
# Generate K positive definite kernel matrices (10x10)
kernmats = np.zeros((ell, ell, K))
for k in range(K):
    A = np.random.rand(ell, ell)
    kernmats[:, :, k] = A @ A.T  # Construct a symmetric PD matrix

Zero Imputation

In [ ]:
def get_model(Q, Lambda):
    """
    Compute the model matrix M from the incomplete kernel matrices.

    Parameters:
        Q (np.ndarray): 3D array of shape (N, N, K) containing K incomplete kernel matrices.
        Lambda (float): Regularization parameter.

    Returns:
        np.ndarray: Model matrix M of shape (N, N).
    """
    N, _, K = Q.shape
    Beta = 1 / (K + Lambda)
    M = Beta * (np.sum(Q, axis=2) + Lambda * np.eye(N))

    if not is_pos_def(M):
        M += 0.001 * np.eye(N)

    return M

Qzero = get_inc_kerns(kernmats, L_v)
Mzero = get_model(Qzero,Lambda)

correlation distance matrix

In [ ]:
diff = 0
for k in range(K):
  diff += 1- ((np.trace(np.matmul(kernmats[:,:,k],Qzero[:,:,k])))/(np.linalg.norm(kernmats[:,:,k], 'fro')*np.linalg.norm(Qzero[:,:,k], 'fro')))
diff = diff/K
print(diff)

0.900040268496329


Mean Imputation

In [ ]:
import numpy as np

def is_pos_def(matrix):
    """Check if a matrix is positive definite."""
    return np.all(np.linalg.eigvals(matrix) > 0)

def mean_svm(Q_inc, L_v):
    """Performs matrix completion by mean-SVM."""
    ell, _, K = Q_inc.shape
    Q = np.zeros((ell, ell, K))

    for k in range(K):
        Qk = Q_inc[:, :, k].copy()
        l_v = L_v[k, :].astype(bool)
        l_h = ~l_v
        Qvv = Qk[np.ix_(l_v, l_v)]

        # Solve for Qvh and Qhh
        vv = Qvv.shape[1]
        Qhh = np.mean(Qvv)
        Qvh = np.mean(Qvv, axis=1).reshape(-1, 1) @ np.ones((1, ell - vv))

        Qk[np.ix_(l_v, l_h)] = Qvh
        Qk[np.ix_(l_h, l_v)] = Qvh.T
        Qk[np.ix_(l_h, l_h)] = Qhh
        Q[:, :, k] = Qk

        if not is_pos_def(Q[:, :, k]):
            Q[:, :, k] += 0.001 * np.eye(ell)

    return Q

Qmean = mean_svm(Qzero, L_v)

In [ ]:
diff_mean = 0
for k in range(K):
  diff_mean += 1- ((np.trace(np.matmul(kernmats[:,:,k],Qmean[:,:,k])))/(np.linalg.norm(kernmats[:,:,k], 'fro')*np.linalg.norm(Qmean[:,:,k], 'fro')))
diff_mean = diff_mean/K
print(diff_mean)

0.0008058609074268844


MKMC-PCA-GK

In [ ]:
def get_UL(S):
    #Returns sorted eigenvectors U and eigenvalues L.""
    eigval, eigvec = np.linalg.eigh(S)
    idx = np.argsort(eigval)[::-1]  # Sort in descending order
    return eigvec[:, idx], eigval[idx]

def mkmc(Qtmp, L_v, Lambda, Mtmp, niters):
    #Completes the kernel matrices using mkmc.
    _, _, K = Qtmp.shape

    for _ in range(niters):
        for k in range(K):
            Qk = Qtmp[:, :, k]
            if not is_pos_def(Qk):
                Qk += 0.001 * np.eye(Qk.shape[0])

            l_v = L_v[k, :]
            l_h = ~l_v
            Qvv = Qk[np.ix_(l_v, l_v)]
            Mvv = Mtmp[np.ix_(l_v, l_v)]
            Mvh = Mtmp[np.ix_(l_v, l_h)]
            Mhh = Mtmp[np.ix_(l_h, l_h)]

            Qvh = Qvv @ np.linalg.solve(Mvv, Mvh)
            Qhh = Mhh - (Mvh.T @ np.linalg.solve(Mvv, Mvh)) + (Mvh.T @ np.linalg.solve(Mvv, Qvv @ np.linalg.solve(Mvv, Mvh)))

            Qk[np.ix_(l_v, l_h)] = Qvh
            Qk[np.ix_(l_h, l_v)] = Qvh.T
            Qk[np.ix_(l_h, l_h)] = Qhh
            Qtmp[:, :, k] = Qk

        Mtmp = get_model(Qtmp, Lambda)

    return Qtmp

def pcagk(Qtmp, Lambda, L_v):
    #Performs PCA-based Gaussian Kernel completion.
    ell, _, _ = Qtmp.shape
    Mtmp = get_model(Qtmp, Lambda)

    for _ in range(10):
        Qpca = mkmc(Qtmp, L_v, Lambda, Mtmp, 1)
        Smat = get_model(Qpca, Lambda)
        U, L = get_UL(Smat)

        q = np.sum(L > np.mean(L))  # Guttman-Kaiser criterion
        sigma2 = np.mean(L[q:])

        W = U[:, :q] @ np.sqrt(np.diag(L[:q]) - sigma2 * np.eye(q))
        Mtmp = W @ W.T + sigma2 * np.eye(ell)
        Qtmp = Qpca

    return Qpca

Qpca  = pcagk(Qzero,Lambda,L_v)

In [ ]:
diff_pca = 0
for k in range(K):
  diff_pca += 1- ((np.trace(np.matmul(kernmats[:,:,k],Qpca[:,:,k])))/(np.linalg.norm(kernmats[:,:,k], 'fro')*np.linalg.norm(Qpca[:,:,k], 'fro')))
diff_pca = diff_pca/K
print(diff_pca)

0.8524647700922916


MKMC - PCA - PSA

In [ ]:
def normalize_kernel(K):
    """
    Normalizes the kernel matrix so that the diagonal entries are 1.

    Parameters:
        K (np.ndarray): A positive definite kernel matrix.

    Returns:
        np.ndarray: The normalized kernel matrix.
    """
    # Compute the square root of the diagonal elements
    diag = np.sqrt(np.diag(K))
    # Outer product of diag with itself
    norm_factor = np.outer(diag, diag)
    # Elementwise division to normalize
    K_normalized = K / norm_factor
    return K_normalized

def compute_kernel_distance(K):
    # Apply the formula; since K is a numpy array, we use it directly.
    D_squared = 2 * (1 - K)
    # Ensure non-negative values (for numerical stability)
    D_squared[D_squared < 0] = 0
    # Take the square root to get the distance matrix
    D = np.sqrt(D_squared)
    return D

In [ ]:
import numpy as np
import gudhi as gd
from scipy.sparse import dok_matrix

def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
    return simplex_tree

def extract_simplices(filtration):
    simplices = {'vertices': [], 'edges': [], 'faces': []}
    for simplex, _ in filtration:
        if len(simplex) == 1:
            simplices['vertices'].append(simplex)
        elif len(simplex) == 2:
            simplices['edges'].append(simplex)
        elif len(simplex) == 3:
            simplices['faces'].append(simplex)
    return simplices

def get_boundary_matrix(simplices_higher, simplices_lower):
    if not simplices_higher or not simplices_lower:
        return None
    n_higher = len(simplices_higher)
    n_lower = len(simplices_lower)
    boundary_matrix = dok_matrix((n_lower, n_higher), dtype=int)
    simplex_to_index = {tuple(simplex): i for i, simplex in enumerate(simplices_lower)}
    for j, higher_simplex in enumerate(simplices_higher):
        for i, vertex in enumerate(higher_simplex):
            boundary_simplex = tuple(sorted(higher_simplex[:i] + higher_simplex[i+1:]))
            if boundary_simplex in simplex_to_index:
                row_index = simplex_to_index[boundary_simplex]
                boundary_matrix[row_index, j] = (-1) ** i
    return boundary_matrix

def compute_laplacian(boundary_prev, boundary_next):
    laplacian = None
    if boundary_prev is not None:
        laplacian = boundary_prev.transpose().dot(boundary_prev)
    if boundary_next is not None:
        if laplacian is None:
            laplacian = boundary_next.dot(boundary_next.transpose())
        else:
            laplacian += boundary_next.dot(boundary_next.transpose())
    return laplacian

def compute_eigenvalues(laplacian):
    if laplacian is not None:
        _, singular_values, _ = np.linalg.svd(laplacian.toarray())
        return np.sort(singular_values)
    return None

def eigenvalues_above_mean(distance_matrix, max_edge_length):
    simplex_tree = compute_rips_from_distance_matrix(distance_matrix, max_edge_length)
    filtration = list(simplex_tree.get_filtration())
    simplices = extract_simplices(filtration)

    boundary_1 = get_boundary_matrix(simplices['edges'], simplices['vertices'])
    laplacian_0 = compute_laplacian(None, boundary_1)

    if laplacian_0 is not None:
        eigenvalues_0 = compute_eigenvalues(laplacian_0)
        mean_eigenvalue = np.mean(eigenvalues_0)
        #count_above_mean = np.sum(eigenvalues_0 > mean_eigenvalue)
        count_above_mean = np.sum(eigenvalues_0 <= 1e-10)
        return mean_eigenvalue, count_above_mean


In [ ]:
import numpy as np

def is_pos_def(matrix):
    return np.all(np.linalg.eigvals(matrix) > 0)

def get_inc_kerns(kernmats, L_v):
    ell, _, K = kernmats.shape
    Q_inc = np.zeros((ell, ell, K))

    for k in range(K):
        l_tmp1 = L_v[k, :]
        Q_inc[np.ix_(l_tmp1, l_tmp1, [k])] = kernmats[np.ix_(l_tmp1, l_tmp1, [k])]
        if not is_pos_def(Q_inc[:, :, k]):
            Q_inc[:, :, k] += 0.001 * np.eye(ell)

    return Q_inc

np.random.seed(42)  # For reproducibility

def get_model(Q, Lambda):
    N, _, K = Q.shape
    Beta = 1 / (K + Lambda)
    M = Beta * (np.sum(Q, axis=2) + Lambda * np.eye(N))

    if not is_pos_def(M):
        M += 0.001 * np.eye(N)

    return M

def get_UL(S):
    eigval, eigvec = np.linalg.eigh(S)
    idx = np.argsort(eigval)[::-1]  # Sort in descending order
    return eigvec[:, idx], eigval[idx]

def mkmc(Qtmp, L_v, Lambda, Mtmp, niters):
    _, _, K = Qtmp.shape
    for _ in range(niters):
        for k in range(K):
            Qk = Qtmp[:, :, k]
            if not is_pos_def(Qk):
                Qk += 0.001 * np.eye(Qk.shape[0])

            l_v = L_v[k, :]
            l_h = ~l_v
            Qvv = Qk[np.ix_(l_v, l_v)]
            Mvv = Mtmp[np.ix_(l_v, l_v)]
            Mvh = Mtmp[np.ix_(l_v, l_h)]
            Mhh = Mtmp[np.ix_(l_h, l_h)]

            Qvh = Qvv @ np.linalg.solve(Mvv, Mvh)
            Qhh = Mhh - (Mvh.T @ np.linalg.solve(Mvv, Mvh)) + (Mvh.T @ np.linalg.solve(Mvv, Qvv @ np.linalg.solve(Mvv, Mvh)))

            Qk[np.ix_(l_v, l_h)] = Qvh
            Qk[np.ix_(l_h, l_v)] = Qvh.T
            Qk[np.ix_(l_h, l_h)] = Qhh
            Qtmp[:, :, k] = Qk

        Mtmp = get_model(Qtmp, Lambda)

    return Qtmp

def pcagk_psa(Qtmp, Lambda, L_v):
    #Performs PCA-based Gaussian Kernel completion.
    ell, _, _ = Qtmp.shape
    Mtmp = get_model(Qtmp, Lambda)

    for _ in range(10):
        Qpca = mkmc(Qtmp, L_v, Lambda, Mtmp, 1)
        Smat = get_model(Qpca, Lambda)
        Smat_norm = normalize_kernel(Smat)
        Smat_dist = compute_kernel_distance(Smat_norm)

        float_value, q = map(lambda x: x.item(), eigenvalues_above_mean(Smat_dist, np.max(Smat_dist)))
        #print(float_value)

        U, L = get_UL(Smat)
        sigma2 = np.mean(L[q:])

        W = U[:, :q] @ np.sqrt(np.diag(L[:q]) - sigma2 * np.eye(q))
        Mtmp = W @ W.T + sigma2 * np.eye(ell)
        Qtmp = Qpca

    return Qpca

Qpca_psa  = pcagk_psa(Qzero,Lambda,L_v)


In [ ]:
diff_psa = 0
for k in range(K):
  diff_psa += 1- ((np.trace(np.matmul(kernmats[:,:,k],Qpca_psa[:,:,k])))/(np.linalg.norm(kernmats[:,:,k], 'fro')*np.linalg.norm(Qpca_psa[:,:,k], 'fro')))
diff_psa = diff_psa/K
print(diff_psa)

0.7421129205951374


In [ ]:
correlation_distance_1 = []
correlation_distance_1.append(diff)
correlation_distance_1.append(diff_mean)
correlation_distance_1.append(diff_pca)
correlation_distance_1.append(diff_psa)

In [ ]:
correlation_distance_1

[np.float64(0.20017922721937265),
 np.float64(0.00030329016076361776),
 np.float64(0.11950258698104983),
 np.float64(0.008342945946313499)]